In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


### <mark>**Stream-Stream Join**</mark>

###### <mark>**Create input folders conceptually and imports**</mark>

In [1]:
# We use Row to create sample records easily in Python
from pyspark.sql import Row

# We use Spark SQL data types to define explicit schemas for streaming
from pyspark.sql.types import (
    StructType, StructField, LongType, StringType, DoubleType
)

# We use these functions later for timestamp conversion and join conditions
from pyspark.sql.functions import col, to_timestamp, expr

StatementMeta(, a008b365-dccd-40c3-ab15-5b9046ff3be1, 3, Finished, Available, Finished, False)

###### <mark>**Step 2: Create sample order event batch**</mark>

In [2]:
# Sample order events stream
# These represent live order creation events arriving from an upstream system

order_events_batch_1 = [
    Row(order_id=910001, customer_id=201, product_id=2001, event_type="order_placed", event_time="2026-04-03 10:00:00", amount=120.50, status="created"),
    Row(order_id=910002, customer_id=202, product_id=2002, event_type="order_placed", event_time="2026-04-03 10:02:00", amount=80.00,  status="created"),
    Row(order_id=910003, customer_id=203, product_id=2003, event_type="order_placed", event_time="2026-04-03 10:05:00", amount=220.00, status="created")
]

order_batch_1_df = spark.createDataFrame(order_events_batch_1)
display(order_batch_1_df)

StatementMeta(, 44b7d0d3-0293-4f1b-821c-f35715663b25, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4d98053d-dd52-45ee-838d-6ab5d84fc0f7)

###### <mark>**Step 3: Write order batch into input folder**</mark>

In [3]:
# Write order events as JSON files into the watched input folder
# This simulates new streaming files arriving

order_batch_1_df.write \
    .mode("append") \
    .json("Files/streaming/order_events_join_input")

StatementMeta(, 44b7d0d3-0293-4f1b-821c-f35715663b25, 5, Finished, Available, Finished, False)

###### <mark>**Step 4: Create sample payment event batch**</mark>

<mark>**Important business note**</mark>

**order_id = 910004 exists in payment stream but not in order stream.
This helps us understand unmatched events later.**

In [4]:
# Sample payment events stream
# These arrive separately from a payment system

payment_events_batch_1 = [
    Row(payment_id=500001, order_id=910001, payment_time="2026-04-03 10:03:00", payment_amount=120.50, payment_status="success", payment_method="card"),
    Row(payment_id=500002, order_id=910002, payment_time="2026-04-03 10:06:00", payment_amount=80.00,  payment_status="success", payment_method="upi"),
    Row(payment_id=500003, order_id=910004, payment_time="2026-04-03 10:07:00", payment_amount=150.00, payment_status="success", payment_method="card")
]

payment_batch_1_df = spark.createDataFrame(payment_events_batch_1)
display(payment_batch_1_df)

StatementMeta(, 44b7d0d3-0293-4f1b-821c-f35715663b25, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b56e1e61-902b-4a03-91b5-a0374fe03c78)

###### <mark>**Step 5: Write payment batch into payment input folder**</mark>

In [5]:
# Write payment events into their own streaming input folder

payment_batch_1_df.write \
    .mode("append") \
    .json("Files/streaming/payment_events_input")

StatementMeta(, 44b7d0d3-0293-4f1b-821c-f35715663b25, 7, Finished, Available, Finished, False)

###### <mark>**Step 6: Define order stream schema**</mark>

In [8]:
# Explicit schema for order event files

order_schema = StructType([
    StructField("order_id", LongType(), True),
    StructField("customer_id", LongType(), True),
    StructField("product_id", LongType(), True),
    StructField("event_type", StringType(), True),
    StructField("event_time", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("status", StringType(), True)
])

StatementMeta(, 44b7d0d3-0293-4f1b-821c-f35715663b25, 10, Finished, Available, Finished, False)

###### <mark>**Step 7: Define payment stream schema**</mark>

In [9]:
# Explicit schema for payment event files

payment_schema = StructType([
    StructField("payment_id", LongType(), True),
    StructField("order_id", LongType(), True),
    StructField("payment_time", StringType(), True),
    StructField("payment_amount", DoubleType(), True),
    StructField("payment_status", StringType(), True),
    StructField("payment_method", StringType(), True)
])

StatementMeta(, 44b7d0d3-0293-4f1b-821c-f35715663b25, 11, Finished, Available, Finished, False)

###### <mark>**Step 8: Read order stream**</mark>

In [10]:
# Streaming read of order event files

order_stream_df = (
    spark.readStream
    .schema(order_schema)
    .json("Files/streaming/order_events_join_input")
)

StatementMeta(, 44b7d0d3-0293-4f1b-821c-f35715663b25, 12, Finished, Available, Finished, False)

###### <mark>**Step 9: Read payment stream**</mark>

In [11]:
# Streaming read of payment event files

payment_stream_df = (
    spark.readStream
    .schema(payment_schema)
    .json("Files/streaming/payment_events_input")
)

StatementMeta(, 44b7d0d3-0293-4f1b-821c-f35715663b25, 13, Finished, Available, Finished, False)

###### <mark>**Step 10: Write both streams to Bronze-style Delta tables**</mark>

In [12]:
# Land raw order stream into Delta
order_bronze_query = (
    order_stream_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "Files/streaming/checkpoints/bronze_order_events_join_stream")
    .trigger(processingTime="10 seconds")
    .toTable("bronze_order_events_join_stream")
)

# Land raw payment stream into Delta
payment_bronze_query = (
    payment_stream_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "Files/streaming/checkpoints/bronze_payment_events_stream")
    .trigger(processingTime="10 seconds")
    .toTable("bronze_payment_events_stream")
)

StatementMeta(, 44b7d0d3-0293-4f1b-821c-f35715663b25, 14, Finished, Available, Finished, False)

###### <mark>**Step 11: Validate both Bronze streams**</mark>

In [13]:
spark.sql("SELECT * FROM bronze_order_events_join_stream ORDER BY order_id").show(truncate=False)
spark.sql("SELECT * FROM bronze_payment_events_stream ORDER BY order_id").show(truncate=False)

StatementMeta(, 44b7d0d3-0293-4f1b-821c-f35715663b25, 15, Finished, Available, Finished, False)

+--------+-----------+----------+------------+-------------------+------+-------+
|order_id|customer_id|product_id|event_type  |event_time         |amount|status |
+--------+-----------+----------+------------+-------------------+------+-------+
|910001  |201        |2001      |order_placed|2026-04-03 10:00:00|120.5 |created|
|910002  |202        |2002      |order_placed|2026-04-03 10:02:00|80.0  |created|
|910003  |203        |2003      |order_placed|2026-04-03 10:05:00|220.0 |created|
+--------+-----------+----------+------------+-------------------+------+-------+

+----------+--------+-------------------+--------------+--------------+--------------+
|payment_id|order_id|payment_time       |payment_amount|payment_status|payment_method|
+----------+--------+-------------------+--------------+--------------+--------------+
|500001    |910001  |2026-04-03 10:03:00|120.5         |success       |card          |
|500002    |910002  |2026-04-03 10:06:00|80.0          |success       |upi   

###### <mark>**Step 12: Read both Bronze tables as streaming sources**</mark>

In [14]:
# Read the Bronze Delta tables again as streaming sources
# This is where the actual stream-stream join begins

order_bronze_stream_df = spark.readStream.table("bronze_order_events_join_stream")
payment_bronze_stream_df = spark.readStream.table("bronze_payment_events_stream")

StatementMeta(, 44b7d0d3-0293-4f1b-821c-f35715663b25, 16, Finished, Available, Finished, False)

###### <mark>**Step 13: Convert event times to timestamp and add watermark**</mark>

In [15]:
# Convert string times into real timestamp type
# Watermark tells Spark how long to wait for late data

order_prepared_df = (
    order_bronze_stream_df
    .withColumn("order_time", to_timestamp(col("event_time")))
    .withWatermark("order_time", "30 minutes")
)

payment_prepared_df = (
    payment_bronze_stream_df
    .withColumn("payment_time_ts", to_timestamp(col("payment_time")))
    .withWatermark("payment_time_ts", "30 minutes")
)

StatementMeta(, 44b7d0d3-0293-4f1b-821c-f35715663b25, 17, Finished, Available, Finished, False)

###### <mark>**Step 14: Stream-stream join condition**</mark>

In [16]:
# Join condition:
# 1. Same order_id
# 2. Payment should happen after order
# 3. Payment should happen within 30 minutes of order

join_condition = (
    (order_prepared_df["order_id"] == payment_prepared_df["order_id"]) &
    (payment_prepared_df["payment_time_ts"] >= order_prepared_df["order_time"]) &
    (payment_prepared_df["payment_time_ts"] <= order_prepared_df["order_time"] + expr("INTERVAL 30 MINUTES"))
)

StatementMeta(, 44b7d0d3-0293-4f1b-821c-f35715663b25, 18, Finished, Available, Finished, False)

###### <mark>**Step 15:Perform stream-stream join**</mark>

In [17]:
# Perform inner join between order and payment streams
# This keeps only matched order-payment pairs

joined_stream_df = order_prepared_df.join(
    payment_prepared_df,
    join_condition,
    "inner"
)

StatementMeta(, 44b7d0d3-0293-4f1b-821c-f35715663b25, 19, Finished, Available, Finished, False)

###### <mark>**Step 16: Select final business columns**</mark>

In [18]:
# Select only meaningful business columns for final output

final_joined_df = joined_stream_df.select(
    order_prepared_df["order_id"],
    order_prepared_df["customer_id"],
    order_prepared_df["product_id"],
    order_prepared_df["order_time"],
    order_prepared_df["amount"].alias("order_amount"),
    payment_prepared_df["payment_id"],
    payment_prepared_df["payment_time_ts"].alias("payment_time"),
    payment_prepared_df["payment_amount"],
    payment_prepared_df["payment_status"],
    payment_prepared_df["payment_method"]
)

StatementMeta(, 44b7d0d3-0293-4f1b-821c-f35715663b25, 20, Finished, Available, Finished, False)

###### <mark>**Step 17: Write joined stream output**</mark>

In [19]:
# Write final joined stream into Delta output table

joined_query = (
    final_joined_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "Files/streaming/checkpoints/silver_order_payment_stream_join")
    .trigger(processingTime="10 seconds")
    .toTable("silver_order_payment_stream_join")
)

StatementMeta(, 44b7d0d3-0293-4f1b-821c-f35715663b25, 21, Finished, Available, Finished, False)

###### <mark>**Step 18: Validate final joined output**</mark>
<mark>**What you should expect**</mark>

Matched rows should appear for:

910001
910002

But not for:

910003 → order exists, no matching payment
910004 → payment exists, no matching order

So this proves the join is working.

In [20]:
spark.sql("""
SELECT *
FROM silver_order_payment_stream_join
ORDER BY order_id
""").show(truncate=False)

StatementMeta(, 44b7d0d3-0293-4f1b-821c-f35715663b25, 22, Finished, Available, Finished, False)

+--------+-----------+----------+-------------------+------------+----------+-------------------+--------------+--------------+--------------+
|order_id|customer_id|product_id|order_time         |order_amount|payment_id|payment_time       |payment_amount|payment_status|payment_method|
+--------+-----------+----------+-------------------+------------+----------+-------------------+--------------+--------------+--------------+
|910001  |201        |2001      |2026-04-03 10:00:00|120.5       |500001    |2026-04-03 10:03:00|120.5         |success       |card          |
|910002  |202        |2002      |2026-04-03 10:02:00|80.0        |500002    |2026-04-03 10:06:00|80.0          |success       |upi           |
+--------+-----------+----------+-------------------+------------+----------+-------------------+--------------+--------------+--------------+

